# Solution E3 · Target encoding before the cross-validation: a transformer inside the Pipeline

**Type**: Find the bug

## Objective

Fix an evaluation inflated by target encoding run on the whole dataset before the cross-validation, by writing a
scikit-learn compatible transformer (`fit`, `transform`, `fit_transform`) and using it inside a `Pipeline`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

df = ds.loans(with_missing=False)
X = df[["occupation"]].copy()
X["noise"] = np.array([f"r{i}" for i in np.random.default_rng(1).integers(0, 300, len(df))])
y = df["defaulted"].to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def te_estimate(cat, y, m):
    d = pd.DataFrame({"c": np.asarray(cat), "y": np.asarray(y, dtype=float)})
    g = d.groupby("c")["y"].agg(["sum", "count"])
    overall = d["y"].mean()
    return ((g["sum"] + m * overall) / (g["count"] + m)).to_dict(), overall


def te_apply(cat, mapping, overall):
    return pd.Series(np.asarray(cat)).map(mapping).fillna(overall).to_numpy(dtype=float)


def te_oof(cat, y, m, k=5, seed=0):
    cat, y = np.asarray(cat), np.asarray(y)
    out = np.zeros(len(cat))
    for tr, va in KFold(k, shuffle=True, random_state=seed).split(cat):
        mapping, overall = te_estimate(cat[tr], y[tr], m)
        out[va] = te_apply(cat[va], mapping, overall)
    return out

## Commented solution

In [ ]:
# the colleague's code: it runs, but it is wrong
Z = np.column_stack([te_apply(X[c], *te_estimate(X[c], y, 5)) for c in X.columns])
auc_bug = cross_val_score(LogisticRegression(), Z, y, cv=cv, scoring="roc_auc").mean()
print(f"colleague's cross-validated AUC: {auc_bug:.3f}")

**Where the bug is.** `te_estimate(X[c], y, 5)` uses the target of **all** the rows, including those that then end up in the
validation fold of the cross-validation: every row is encoded with a mean that contains its own target. For the noise
column (300 levels, about 5 rows each) the encoding is almost the target itself, and the AUC goes up as if the noise were a
signal. The fix: the encoder must sit **inside** the pipeline, so that at every fold the mapping is computed only on the
training part of the fold.

In [ ]:
class MyTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, m=5):
        self.m = m

    def fit(self, X, y):
        X = pd.DataFrame(X)
        self.columns_ = list(X.columns)
        self.maps_ = {c: te_estimate(X[c], y, self.m) for c in self.columns_}
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return np.column_stack([te_apply(X[c], *self.maps_[c]) for c in self.columns_])

    def fit_transform(self, X, y):
        self.fit(X, y)                                    # the complete mappings are needed for transform on the validation set
        X = pd.DataFrame(X)
        return np.column_stack([te_oof(X[c], y, self.m) for c in self.columns_])     # but the training set is encoded out-of-fold

In [ ]:
pipe = make_pipeline(MyTargetEncoder(m=5), LogisticRegression())
auc_ok = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()
noise_only_auc = cross_val_score(pipe, X[["noise"]], y, cv=cv, scoring="roc_auc").mean()
print(f"cross-validated AUC: colleague {auc_bug:.3f} | correct pipeline {auc_ok:.3f} | noise only {noise_only_auc:.3f}")

## Check

In [ ]:
with verify():
    check(auc_bug > 0.75, "the colleague's number is very high: it is the target leaking", "auc_bug should exceed 0.75: don't modify the colleague's code")
    check(0.55 < auc_ok < 0.72, "with the pipeline the AUC drops to a realistic value (between 0.55 and 0.72)",
          "auc_ok must come from cross_val_score on make_pipeline(MyTargetEncoder(m=5), LogisticRegression())")
    check(noise_only_auc < 0.6, "on the noise alone the cross-validated AUC is close to 0.5", "on the noise alone with the pipeline the AUC must stay below 0.6")
    check(isinstance(MyTargetEncoder(), TransformerMixin), "the transformer is scikit-learn compatible", "inherit from BaseEstimator and TransformerMixin")
    enc = MyTargetEncoder(m=5).fit(X, y)
    new = enc.transform(pd.DataFrame({"occupation": ["occ_zz"], "noise": ["r_new"]}))
    check(np.isfinite(new).all() and close(new, [[y.mean(), y.mean()]]), "unknown categories get the global mean",
          "a never-seen category must be encoded with the global mean of the target")
    check(not np.allclose(MyTargetEncoder(m=5).fit_transform(X, y), enc.transform(X)),
          "fit_transform (out-of-fold) is different from fit + transform on the same training set",
          "fit_transform must use te_oof; transform uses the mapping of the whole training set")
done()

## What you learned

- A `Pipeline` calls `fit_transform(X_train, y_train)` on the training folds and `transform(X_val)` on the validation folds:
  overriding `fit_transform` with the out-of-fold encoding gives the right behavior. The bug is repeating the
  same mapping on everything.
- The colleague's AUC (about 0.85) is a mirage: with the pipeline it drops to about 0.6, the real value of the occupation. On the noise
  alone it falls from high values to about 0.5.
- Writing a compatible transformer is little code: `BaseEstimator` for `get_params`/`clone`, `TransformerMixin` for the default
  `fit_transform`, `fit` that returns `self`. Module 3 uses this pattern for custom preprocessors.

## Common pitfall

Relying on the default `fit_transform` (`TransformerMixin` calls `fit(X, y).transform(X)`): in target encoding it means
encoding the training set with the in-sample mapping, that is the very bug you were trying to avoid. For transformers that use `y` the
`fit_transform` must be overridden.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why must a transformer that uses the target be written with a `fit_transform` different from `fit` + `transform`?</summary>

Because in training the encoding of every row must not contain its target: `fit_transform` computes it out-of-fold. In
validation or in production, instead, the rows are new and the mapping of the whole training set is used (`transform`). If `fit_transform`
were `fit` + `transform` on the same training set, the model would see a feature that has already seen the target and would learn to trust it
too much: in production it will not be like that.

</details>